# Python client for DB Service

The DB Service Python client provides a thin wrapper over the DB Service APIs, making it easier to connect to a running service and perform client operations from Python.

Use this client to create a session, run queries, manage tables, and interact with DB Service from Python.

## Connect

Create a session to connect to DB Service:

In [1]:
import dbservice_client as dbs
import pandas as pd  # Query examples return dataframes
from datetime import datetime as dt, timezone, timedelta

# Default endpoint: localhost:8080
session = dbs.Session()

# Optional explicit endpoint
rest_session = dbs.Session(endpoint="localhost:8080")

## Managing Tables
Use these calls to define and inspect table schemas in DB Service.

In [2]:
# List tables (empty to begin with)
session.list_tables()

[]

### Reference data and foreign keys

`instruments` is a reference table: a small, slowly changing table of instrument metadata, keyed on `sym` using `primaryKeys`. List the key column first in `columns` so that the schema matches the column order of the keyed table.

Declaring `"foreign": "instruments.sym"` on the `fxquote.sym` column links the quotes to that reference data, which lets a query read `instruments` columns using dot notation, for example `instruments.category`.

In [3]:
# Create the 'instruments' reference table, keyed on 'sym'
session.create_table(
    table="instruments",
    type="splayed",
    primaryKeys=["sym"],
    columns=[
        {"name": "sym", "type": "symbol"},
        {"name": "instrumentid", "type": "long"},
        {"name": "category", "type": "symbol"},
        {"name": "decimals", "type": "long"},
        {"name": "pipdecimals", "type": "long"},
    ]
)

{'jobId': '93202562-31a4-7e76-42be-0bb1b8ad5a10',
 'status': 'completed',
 'statusUri': '/api/v0/jobs/93202562-31a4-7e76-42be-0bb1b8ad5a10',
 'startedAt': '2026-09-02T15:35:05.517497793',
 'finishedAt': None,
 'table': [],
 'warnings': []}

In [4]:
# Create partitioned table ('fxquote'), with 'sym' as a foreign key into 'instruments'
session.create_table(
    table="fxquote",
    type="partitioned",
    prtnCol="ts",
    sortColsDisk=["sym"],
    sortColsOrd=["sym"],
    columns=[
        {"name": "trddate", "type": "date"},
        {"name": "ts", "type": "timestamp"},
        {"name": "sym", "type": "symbol", "foreign": "instruments.sym", "attrMem": "grouped", "attrDisk": "parted", "attrOrd": "parted"},
        {"name": "bid", "type": "float"},
        {"name": "ask", "type": "float"},
    ]
)

{'jobId': 'd7742396-3068-6ebc-17cf-cf7dcd1d3274',
 'status': 'completed',
 'statusUri': '/api/v0/jobs/d7742396-3068-6ebc-17cf-cf7dcd1d3274',
 'startedAt': '2026-09-02T15:35:14.589340139',
 'finishedAt': None,
 'table': [],
 'warnings': []}

In [5]:
# List tables ('fxquote' and 'instruments' tables returned)
session.list_tables()

['instruments', 'fxquote']

In [6]:
# Describe the 'fxquote' table
session.describe_table(table="fxquote")

{'columns': [{'name': 'trddate', 'type': 'date'},
  {'name': 'ts', 'type': 'timestamp'},
  {'name': 'sym',
   'type': 'symbol',
   'foreign': 'instruments.sym',
   'attrMem': 'grouped',
   'attrDisk': 'parted',
   'attrOrd': 'parted'},
  {'name': 'bid', 'type': 'float'},
  {'name': 'ask', 'type': 'float'}],
 'type': 'partitioned',
 'prtnCol': 'ts',
 'sortColsOrd': ['sym'],
 'sortColsDisk': ['sym'],
 'name': 'fxquote'}

## Importing Data
DB Service supports both `file-based` and `in-memory` ingest. Any file you want to import must first be copied into the DB Service `imports` staging directory, for example: `~/.kx/db-service/data/imports/`

### Import CSV

In [7]:
# Import a CSV file into the existing 'fxquote' table
job = session.import_files(table="fxquote", path="fxquote.csv.gz")

In [8]:
# Check the status of the above import job
session.get_import(job_id=job["jobId"])

{'jobId': 'ddefbc19-498b-14cd-b1be-b86b91232722',
 'database': 'db',
 'jobType': 'import',
 'status': 'completed',
 'affectedTables': ['fxquote'],
 'processedPartitions': ['2026-03-02'],
 'progress': {'currentPartition': '',
  'partitionIndex': 1,
  'partitionTotal': 1,
  'currentTable': '',
  'tableIndex': 0,
  'tableTotal': 0},
 'error': '',
 'warnings': [],
 'updated': '2026-09-02T15:35:32.685108738'}

In [9]:
# Import a parquet file into the existing 'fxquote' table
job = session.import_files(table='fxquote', path='fxquote.parquet')

In [10]:
# Import a CSV file into the 'instruments' reference table created above.
# (createTable=True would create a missing table from the file, but a reference
#  table needs the primary key that only an explicit schema can declare.)
job = session.import_files(table="instruments", path="instruments.csv")

### Import JSON
Users can import data directly from Python without file staging.

In [11]:
# Objects payload imported to 'instruments' table
job = session.import_data(
    table="instruments",
    data=[
        {"instrumentid": 77, "sym": "USDBRL", "category": "EM", "decimals": 4, "pipdecimals": 4},
        {"instrumentid": 78, "sym": "USDKRW", "category": "EM", "decimals": 2, "pipdecimals": 2},
    ],
    insert_as="objects",
)

In [12]:
# Rows payload imported to the 'fxquote' table
job = session.import_data(
    table="fxquote",
    data=[
        ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
        ["2026-01-21", "2026-01-21T10:00:00.000", "EURUSD", 901.2, 901.3],
    ],
    columnNames=["trddate", "ts", "sym", "bid", "ask"],
    insert_as="rows",
)
# Note: for rows payload, columnNames are required.

## Querying Tables
Run structured, SQL, or q queries against DB Service.

In [13]:
# Structured query
session.query_simple(
    table="fxquote",
    startTS="2026.03.02D00:00:00.000",
    endTS="2026.03.03D00:00:00.000",
    sortCols=["ts"],
    limit=5,
    return_as="json",
)

[{'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'AUDUSD',
  'bid': 0.67091,
  'ask': 0.67094},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'EURUSD',
  'bid': 1.16397,
  'ask': 1.16399},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'GBPUSD',
  'bid': 1.3419,
  'ask': 1.34194},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'USDCAD',
  'bid': 1.38744,
  'ask': 1.3875},
 {'trddate': '2026-03-02',
  'ts': '2026-03-02T00:00:00.000000000',
  'sym': 'USDJPY',
  'bid': 158.162,
  'ask': 158.167}]

A foreign key lets a structured query reach into reference data using `table.column` dot notation. Dot columns can be used in `agg`, `groupBy` and `filter`, and are returned under their dotted name.

In [14]:
# Structured query joining reference data over the 'sym' foreign key:
# 'instruments.category' is returned alongside the quotes, and is filtered on 'Major'
session.query_simple(
    table="fxquote",
    startTS="2026.03.02D00:00:00.000",
    endTS="2026.03.02D00:00:10.000",
    agg=["ts", "sym", "bid", "ask", "instruments.category"],
    filter=[["=", "instruments.category", "Major"]],
    sortCols=["ts"],
    return_as="pandas",
)

,ts,sym,bid,ask,instruments.category
0,2026-03-02T00:00:00.000000000,EURUSD,1.16397,1.16399,Major
1,2026-03-02T00:00:00.000000000,GBPUSD,1.34190,1.34194,Major
2,2026-03-02T00:00:00.000000000,USDCAD,1.38744,1.38750,Major
3,2026-03-02T00:00:00.000000000,USDJPY,158.16200,158.16700,Major
4,2026-03-02T00:00:01.000000000,GBPUSD,1.34189,1.34194,Major
5,2026-03-02T00:00:01.000000000,USDJPY,158.16300,158.17000,Major
6,2026-03-02T00:00:02.000000000,USDJPY,158.16600,158.17000,Major
7,2026-03-02T00:00:03.000000000,GBPUSD,1.34187,1.34191,Major
8,2026-03-02T00:00:06.000000000,USDCAD,1.38741,1.38746,Major
9,2026-03-02T00:00:06.000000000,USDJPY,158.16300,158.16500,Major


In [15]:
# SQL query
session.query_sql(
    query="SELECT * FROM instruments WHERE category LIKE 'EM'",
    return_as="pandas",
)

,sym,instrumentid,category,decimals,pipdecimals
0,CHFZAR,14,EM,5,4
1,EURTRY,29,EM,5,4
2,EURZAR,31,EM,5,4
3,GBPZAR,41,EM,5,4
4,USDCNH,61,EM,5,4
5,USDINR,66,EM,5,4
6,USDMXN,68,EM,5,4
7,USDTHB,73,EM,3,2
8,USDTRY,74,EM,5,4
9,USDZAR,75,EM,5,4


In [16]:
# QSQL query
session.query_q(
    query='select o:first bid,h:max bid,l:min bid,c:last bid by trddate,sym from fxquote',
    return_as="pandas",
)

,trddate,sym,o,h,l,c
0,2026-01-21,EURUSD,901.20000,901.20000,901.20000,901.20000
1,2026-03-02,AUDUSD,0.67091,0.67469,0.67065,0.67307
2,2026-03-02,EURUSD,1.16397,1.17680,1.16325,1.17268
3,2026-03-02,GBPUSD,1.34190,1.34913,1.34101,1.34404
4,2026-03-02,USDCAD,1.38744,1.38790,1.38140,1.38336
5,2026-03-02,USDJPY,158.16200,158.60200,157.47600,158.15100
6,2026-03-03,AUDUSD,0.67310,0.67781,0.67273,0.67540
7,2026-03-03,EURUSD,1.17258,1.17430,1.16703,1.16726
8,2026-03-03,GBPUSD,1.34401,1.34588,1.33996,1.34176
9,2026-03-03,USDCAD,1.38338,1.38441,1.37855,1.38440


**Return format:** `return_as` may be `json`, `pandas`, or `pykx`. If omitted, it defaults to `json`.

## Deleting tables
A table that is the target of a foreign key cannot be dropped while the referencing table still exists, so drop `fxquote` before `instruments`.

In [17]:
# List tables (expected: 'fxquote' and 'instruments')
session.list_tables()

['instruments', 'fxquote']

In [18]:
# Drop the 'fxquote' table (the table holding the foreign key)
session.drop_table(table="fxquote")

{'jobId': 'a4caebf2-9357-7b1b-98c7-7bfe8d07f39f',
 'status': 'completed',
 'statusUri': '/api/v0/jobs/a4caebf2-9357-7b1b-98c7-7bfe8d07f39f',
 'startedAt': '2026-09-02T15:37:38.783934395',
 'finishedAt': None,
 'table': [],
 'warnings': []}

In [19]:
# Drop the 'instruments' table
session.drop_table(table="instruments")

{'jobId': 'bcc3367c-5ebd-b8ae-2217-896fa3bac031',
 'status': 'completed',
 'statusUri': '/api/v0/jobs/bcc3367c-5ebd-b8ae-2217-896fa3bac031',
 'startedAt': '2026-09-02T15:37:47.766282314',
 'finishedAt': None,
 'table': [],
 'warnings': []}

In [20]:
# List tables (expected: both tables are gone)
session.list_tables()

[]